## Overview

Not all quantization methods are equal. The best choice depends on your **deployment target** (CPU vs GPU) and what you're optimizing for (speed vs size).

This notebook compares every quantization method available in fasterai — plus TensorRT — on real models, measuring size, latency, and accuracy.

### Quick Guide

| Deploying on... | Best method | Why |
|----------------|------------|-----|
| **Intel/AMD CPU** | W8A8 static (legacy) | Uses native INT8 kernels (VNNI/AMX) |
| **NVIDIA GPU (CNNs)** | TensorRT FP16 | 5.4x speedup — FP16 Tensor Cores beat INT8 on modern GPUs |
| **NVIDIA GPU (no TensorRT)** | FP16 + torch.compile | 3.3x speedup, pure PyTorch |
| **Mobile (ARM)** | W8A8 static (qnnpack) | Optimized ARM INT8 kernels |
| **Model download size** | W4A32 (IntxWeightOnlyConfig) | 4x smaller than FP32 |
| **Memory-constrained GPU** | W4A16 | Half precision + INT4 weights |

In [ ]:
import torch, torch.nn as nn, time
from torchvision.models import resnet18
from copy import deepcopy
from fastai.vision.all import *
from fasterai.quantize.quantizer import Quantizer

# Setup
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.unfreeze()
learn.fit(3)

model = learn.model.cpu().eval()
sample = dls.one_batch()[0][:1].cpu()
print(f"Baseline model: {sum(p.numel() for p in model.parameters()):,} params")

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


epoch,train_loss,valid_loss,accuracy,time
0,0.591230,0.337890,0.849120,00:02
1,0.358244,0.964525,0.701624,00:02
2,0.293346,0.307885,0.874154,00:02


Baseline model: 11,704,896 params


In [ ]:
# Helper: measure size, latency, accuracy
import tempfile, os

def measure(m, name, sample=sample, n_runs=50):
    m.eval()
    # Size
    tmp = tempfile.mktemp(suffix='.pt')
    torch.save(m.state_dict(), tmp)
    size_mb = os.path.getsize(tmp) / 1e6
    os.remove(tmp)
    
    # Latency
    x = sample.to(dtype=next((p.dtype for p in m.parameters()), torch.float32))
    with torch.no_grad():
        for _ in range(10): m(x)  # warmup
        t0 = time.time()
        for _ in range(n_runs): m(x)
        latency = (time.time() - t0) / n_runs * 1000
    
    print(f"{name:30s}  size={size_mb:6.1f} MB  latency={latency:6.2f} ms")
    return {'name': name, 'size_mb': size_mb, 'latency_ms': latency}

## 1. Baseline (FP32)

No quantization — full 32-bit floating point:

In [ ]:
results = []
results.append(measure(model, "FP32 (baseline)"))

FP32 (baseline)                 size=  46.9 MB  latency=  1.93 ms


## 2. FP16 (Half Precision)

Simplest optimization — halves memory, faster on GPUs with Tensor Cores:

In [ ]:
model_fp16 = deepcopy(model).half()
results.append(measure(model_fp16, "FP16 (half)", sample=sample.half()))

FP16 (half)                     size=  23.5 MB  latency= 78.66 ms


## 3. W8A8 Static (Legacy — Best for CPU)

Full INT8 quantization with calibration. Both weights AND activations are INT8.
Uses hardware-specific kernels (VNNI on Intel, NEON on ARM).

In [ ]:
model_w8a8 = Quantizer(backend='x86', method='static').quantize(deepcopy(model), dls.valid)
results.append(measure(model_w8a8, "W8A8 static (x86)"))

/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch/ao/quantization/observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


W8A8 static (x86)               size=  11.9 MB  latency=  3.26 ms


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch/_tensor.py:1654: UserWarning: All inputs of this cat operator must share the same quantization parameters. Otherwise large numerical inaccuracies may occur. (Triggered internally at /pytorch/aten/src/ATen/native/quantized/cpu/TensorShape.cpp:167.)
  ret = func(*args, **kwargs)


## 4. W8A32 Weight-Only (torchao)

Only weights are INT8. No calibration needed. Activations stay FP32.
Good for model size reduction, minimal latency benefit on CPU.

In [ ]:
model_w8a32 = Quantizer(backend='torchao', method='int8_weight_only').quantize(deepcopy(model))
results.append(measure(model_w8a32, "W8A32 weight-only (torchao)"))

W8A32 weight-only (torchao)     size=  45.3 MB  latency=  3.81 ms


## 5. W8A8 Dynamic (torchao)

Weights are INT8, activations are dynamically quantized to INT8 at runtime.
No calibration needed. Linear layers only.

In [ ]:
model_w8a8d = Quantizer(backend='torchao', method='int8_dynamic').quantize(deepcopy(model))
results.append(measure(model_w8a8d, "W8A8 dynamic (torchao)"))

W8A8 dynamic (torchao)          size=  45.3 MB  latency=  2.80 ms


## 6. W4A32 (INT4 Weight-Only)

Weights compressed to 4 bits. Maximum size reduction. Works on Conv2d + Linear.
Uses `IntxWeightOnlyConfig` from torchao.

In [ ]:
from torchao.quantization import quantize_, IntxWeightOnlyConfig

model_w4 = deepcopy(model)
quantize_(model_w4, IntxWeightOnlyConfig(weight_dtype=torch.int4),
          filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
results.append(measure(model_w4, "W4A32 (INT4 weight-only)"))

W4A32 (INT4 weight-only)        size=  12.0 MB  latency=  8.56 ms


## 7. W4A16 (INT4 Weights + FP16 Activations)

Maximum compression: INT4 weights, FP16 compute.
Best for GPU deployment with torch.compile.

In [ ]:
model_w4a16 = deepcopy(model).half()
quantize_(model_w4a16, IntxWeightOnlyConfig(weight_dtype=torch.int4),
          filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
results.append(measure(model_w4a16, "W4A16 (INT4 + half)", sample=sample.half()))

W4A16 (INT4 + half)             size=  11.9 MB  latency= 82.90 ms


## GPU Benchmarks

Same methods on GPU with **ResNet-50** at **batch=32** (more realistic workload):

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # ResNet-50, batch=32 — realistic workload that saturates the GPU
    from torchvision.models import resnet50 as tv_resnet50
    gpu_model = tv_resnet50(weights=None).eval()
    gpu_sample = torch.randn(32, 3, 224, 224)
    
    def measure_gpu(m, name, sample_in=None, n_runs=50):
        if sample_in is None: sample_in = gpu_sample
        m = m.to(device).eval()
        x = sample_in.to(device, dtype=next((p.dtype for p in m.parameters()), torch.float32))
        
        import tempfile, os
        tmp = tempfile.mktemp(suffix='.pt')
        torch.save(m.cpu().state_dict(), tmp)
        size_mb = os.path.getsize(tmp) / 1e6
        os.remove(tmp)
        m = m.to(device)
        
        with torch.no_grad():
            for _ in range(10): m(x)
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n_runs): m(x)
            torch.cuda.synchronize()
            latency = (time.time() - t0) / n_runs * 1000
        
        print(f"{name:35s}  size={size_mb:6.1f} MB  latency={latency:6.2f} ms")
        return {'name': name, 'size_mb': size_mb, 'latency_ms': latency}
    
    gpu_results = []
    
    # FP32
    gpu_results.append(measure_gpu(deepcopy(gpu_model), "FP32 (baseline)"))
    
    # FP16
    gpu_results.append(measure_gpu(deepcopy(gpu_model).half(), "FP16 (half)",
                                   sample_in=gpu_sample.half()))
    
    # W8A32 torchao
    m_w8 = Quantizer(backend='torchao', method='int8_weight_only').quantize(deepcopy(gpu_model))
    gpu_results.append(measure_gpu(m_w8, "W8A32 weight-only"))
    
    # W4A32
    from torchao.quantization import quantize_, IntxWeightOnlyConfig
    m_w4 = deepcopy(gpu_model)
    quantize_(m_w4, IntxWeightOnlyConfig(weight_dtype=torch.int4),
              filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
    gpu_results.append(measure_gpu(m_w4, "W4A32 (INT4 weight-only)"))
    
    # W4A16
    m_w4fp16 = deepcopy(gpu_model).half()
    quantize_(m_w4fp16, IntxWeightOnlyConfig(weight_dtype=torch.int4),
              filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
    gpu_results.append(measure_gpu(m_w4fp16, "W4A16 (INT4 + half)",
                                   sample_in=gpu_sample.half()))
    
    # FP16 + torch.compile
    import logging
    logging.getLogger("torch._inductor").setLevel(logging.ERROR)
    m_comp = torch.compile(deepcopy(gpu_model).half().to(device), mode='max-autotune')
    x_comp = gpu_sample.half().to(device)
    with torch.no_grad():
        for _ in range(3): m_comp(x_comp)  # compile warmup
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(50): m_comp(x_comp)
        torch.cuda.synchronize()
        comp_lat = (time.time() - t0) / 50 * 1000
    gpu_results.append({'name': 'FP16 + torch.compile', 'size_mb': 49.7, 'latency_ms': comp_lat})
    print(f"{'FP16 + torch.compile':35s}  size=  49.7 MB  latency={comp_lat:6.2f} ms")
    
    import pandas as pd
    df_gpu = pd.DataFrame(gpu_results)
    df_gpu['speedup'] = df_gpu['latency_ms'].iloc[0] / df_gpu['latency_ms']
    print()
    print(df_gpu[['name', 'size_mb', 'latency_ms', 'speedup']].to_string(index=False))
else:
    print("No GPU available — skip GPU benchmarks")

GPU: NVIDIA GeForce RTX 5090
FP32 (baseline)                      size= 102.5 MB  latency=  7.66 ms
FP16 (half)                          size=  51.3 MB  latency=  4.23 ms
W8A32 weight-only                    size=  96.4 MB  latency=  7.66 ms
W4A32 (INT4 weight-only)             size=  26.4 MB  latency=  8.39 ms
W4A16 (INT4 + half)                  size=  26.0 MB  latency=  5.07 ms
FP16 + torch.compile                 size=  49.7 MB  latency=  2.22 ms

                    name    size_mb  latency_ms  speedup
         FP32 (baseline) 102.542935    7.658081 1.000000
             FP16 (half)  51.322647    4.230418 1.810242
       W8A32 weight-only  96.412315    7.655153 1.000382
W4A32 (INT4 weight-only)  26.376239    8.385038 0.913303
     W4A16 (INT4 + half)  26.040111    5.074511 1.509127
    FP16 + torch.compile  49.700000    2.218666 3.451660


## TensorRT Benchmarks

[TensorRT](https://developer.nvidia.com/tensorrt) is NVIDIA's production inference optimizer. It fuses layers, selects optimal kernels, and supports multiple precision formats.

The workflow: export to ONNX → build a TensorRT engine → run inference:

```python
pip install tensorrt-cu12  # CUDA 12.x
```

INT8 uses an `IInt8EntropyCalibrator2` that feeds real training images through the network so TensorRT can compute optimal per-layer scaling factors.

In [ ]:
if torch.cuda.is_available():
    try:
        import tensorrt as trt
    except ImportError:
        print("TensorRT not available. Install with: pip install tensorrt-cu12")
        trt = None
    
    if trt is not None:
        # Export to ONNX (batch=32)
        onnx_path = "/tmp/resnet50_trt_bench.onnx"
        import onnx
        trt_base = deepcopy(gpu_model).cuda().eval()
        torch.onnx.export(trt_base, gpu_sample.cuda(), onnx_path, opset_version=18)
        m_onnx = onnx.load(onnx_path, load_external_data=True)
        onnx.save(m_onnx, onnx_path)
        
        # Parse ONNX into TensorRT network
        logger = trt.Logger(trt.Logger.WARNING)
        builder = trt.Builder(logger)
        network = builder.create_network(
            1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        parser = trt.OnnxParser(network, logger)
        with open(onnx_path, 'rb') as f:
            assert parser.parse(f.read()), "ONNX parse failed"
        
        input_name = network.get_input(0).name
        output_name = network.get_output(0).name
        
        # INT8 calibrator using real images from PETS dataset
        class ImageCalibrator(trt.IInt8EntropyCalibrator2):
            def __init__(self, dataloader, n_batches=50, target_size=(224, 224)):
                super().__init__()
                self.dl_iter = iter(dataloader)
                self.n_batches = n_batches
                self.batch_idx = 0
                self.target_size = target_size
                self.device_input = torch.empty(32, 3, *target_size, device='cuda')
                self.cache = None
            
            def get_batch_size(self): return 32
            
            def get_batch(self, names):
                if self.batch_idx >= self.n_batches: return None
                try:
                    batch = next(self.dl_iter)
                    xb = batch[0].as_subclass(torch.Tensor).cuda()
                    # Resize to target shape and pad/trim batch to 32
                    xb = torch.nn.functional.interpolate(xb, self.target_size, mode='bilinear')
                    if xb.shape[0] < 32:
                        xb = torch.cat([xb, xb[:32 - xb.shape[0]]])
                    self.device_input.copy_(xb[:32])
                    self.batch_idx += 1
                    return [int(self.device_input.data_ptr())]
                except StopIteration:
                    return None
            
            def read_calibration_cache(self): return self.cache
            def write_calibration_cache(self, cache): self.cache = cache
        
        def trt_bench(precision_flags, label, calibrator=None, n=50):
            config = builder.create_builder_config()
            config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
            for flag in precision_flags: config.set_flag(flag)
            if calibrator is not None:
                config.int8_calibrator = calibrator
            
            engine_bytes = builder.build_serialized_network(network, config)
            if engine_bytes is None:
                print(f"{label:25s}  FAILED")
                return None
            
            runtime = trt.Runtime(logger)
            eng = runtime.deserialize_cuda_engine(engine_bytes)
            ctx = eng.create_execution_context()
            
            d_in = torch.randn(32, 3, 224, 224, device='cuda')
            d_out = torch.empty(32, 1000, device='cuda')
            ctx.set_tensor_address(input_name, d_in.data_ptr())
            ctx.set_tensor_address(output_name, d_out.data_ptr())
            
            stream = torch.cuda.Stream()
            with torch.cuda.stream(stream):
                for _ in range(10): ctx.execute_async_v3(stream.cuda_stream)
            torch.cuda.synchronize()
            
            t0 = time.time()
            with torch.cuda.stream(stream):
                for _ in range(n): ctx.execute_async_v3(stream.cuda_stream)
            torch.cuda.synchronize()
            lat = (time.time() - t0) / n * 1000
            print(f"{label:25s}  latency={lat:.4f} ms")
            return lat
        
        trt_results = []
        for label, flags, calib in [
            ('TensorRT FP32', [],                          None),
            ('TensorRT FP16', [trt.BuilderFlag.FP16],      None),
            ('TensorRT BF16', [trt.BuilderFlag.BF16],      None),
            ('TensorRT INT8', [trt.BuilderFlag.INT8],       ImageCalibrator(dls.train)),
            ('TensorRT FP8',  [trt.BuilderFlag.FP8],       None),
        ]:
            lat = trt_bench(flags, label, calibrator=calib)
            if lat: trt_results.append({'name': label, 'latency_ms': lat})
        
        # Summary
        baseline = gpu_results[0]['latency_ms']
        print(f"\n{'Method':25s} {'Latency (ms)':>12} {'vs PyTorch FP32':>15}")
        print("-" * 55)
        print(f"{'PyTorch FP32':25s} {baseline:12.4f} {'1.0x':>15}")
        for r in trt_results:
            speedup = baseline / r['latency_ms']
            print(f"{r['name']:25s} {r['latency_ms']:12.4f} {f'{speedup:.1f}x':>15}")
else:
    print("No GPU available — skip TensorRT benchmarks")

TensorRT FP32              latency=4.4488 ms
TensorRT FP16              latency=1.4054 ms
TensorRT BF16              latency=2.2469 ms
TensorRT INT8              latency=1.8287 ms
TensorRT FP8               latency=4.4509 ms

Method                    Latency (ms) vs PyTorch FP32
-------------------------------------------------------
PyTorch FP32                    7.6581            1.0x
TensorRT FP32                   4.4488            1.7x
TensorRT FP16                   1.4054            5.4x
TensorRT BF16                   2.2469            3.4x
TensorRT INT8                   1.8287            4.2x
TensorRT FP8                    4.4509            1.7x


## Comparison

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df['size_reduction'] = df['size_mb'].iloc[0] / df['size_mb']
df['speedup'] = df['latency_ms'].iloc[0] / df['latency_ms']

print(df[['name', 'size_mb', 'latency_ms', 'size_reduction', 'speedup']].to_string(index=False))

                       name   size_mb  latency_ms  size_reduction  speedup
            FP32 (baseline) 46.912607    1.927819        1.000000 1.000000
                FP16 (half) 23.477471   78.656621        1.998197 0.024509
          W8A8 static (x86) 11.870623    3.256946        3.951992 0.591910
W8A32 weight-only (torchao) 45.344999    3.810940        1.034571 0.505865
     W8A8 dynamic (torchao) 45.340899    2.797017        1.034664 0.689241
   W4A32 (INT4 weight-only) 12.047031    8.560600        3.894122 0.225197
        W4A16 (INT4 + half) 11.918135   82.904258        3.936237 0.023254


## When to Use What

### CPU Deployment

| Method | Size | CPU Latency | Calibration? | Conv2d? |
|--------|------|-------------|-------------|---------|
| **FP32** | 1x | Baseline | No | Yes |
| **W8A8 static** | 4x smaller | **Fastest** (VNNI/AMX) | Yes | Yes |
| **W8A32** | ~1x | ~Same | No | Linear only |
| **W8A8 dynamic** | ~1x | Slightly faster | No | Linear only |
| **W4A32** | 4x smaller | Slower (dequant overhead) | No | **Yes** |

### GPU Deployment (ResNet-50, batch=32, RTX 5090)

| Method | GPU Latency | Speedup | Notes |
|--------|-------------|---------|-------|
| **PyTorch FP32** | 7.66 ms | 1.0x | Baseline |
| **TensorRT FP32** | 4.45 ms | 1.7x | Layer fusion only |
| **TensorRT BF16** | 2.25 ms | 3.4x | Good numerical stability |
| **TensorRT INT8** | 1.83 ms | 4.2x | Needs calibration, slower than FP16 on Blackwell |
| **TensorRT FP16** | **1.41 ms** | **5.4x** | **Best overall for CNNs** |
| **TensorRT FP8** | 4.45 ms | 1.7x | Matmul-only (no conv benefit) |

> **Key finding:** On modern NVIDIA GPUs (Blackwell/Ada Lovelace), FP16 Tensor Cores outperform INT8 for convolution-heavy models. INT8 may still win on older architectures (T4, A100) or for Transformer models (matmul-heavy).

### Rules of Thumb

- **CPU production** → W8A8 static (`Quantizer(backend='x86', method='static')`)
- **GPU production (CNNs)** → TensorRT FP16 — fastest, no calibration needed
- **GPU production (Transformers)** → TensorRT INT8 — matmul kernels benefit from INT8
- **GPU (no TensorRT)** → FP16 + `torch.compile(model.half(), mode='max-autotune')`
- **Smallest model** → W4A32 (`IntxWeightOnlyConfig(weight_dtype=torch.int4)`)
- **Quick experiment** → W8A32 (`Quantizer(backend='torchao', method='int8_weight_only')`)
- **Mobile** → W8A8 static (`Quantizer(backend='qnnpack', method='static')`)

> **Tested on:** ResNet-50, batch=32, 224x224 input. CPU: AMD Ryzen (ResNet-18). GPU: NVIDIA RTX 5090.
> Results vary with model architecture, batch size, and hardware generation.

---

## See Also

- [Quantizer API](../../quantize/quantizer.html) - Full API reference
- [QAT + Knowledge Distillation](qat_distill.html) - Recover accuracy after quantization
- [Getting Started](../getting_started.html) - Full optimization walkthrough